# Landslide Safety Guide - RAG Walkthrough

This notebook shows how the project answers questions using its safety PDF. **RAG** means Retrieval-Augmented Generation: first search the document for useful passages, then ask a language model to answer using those passages.

The source of the implementation is `app/observations/rag.py`. Django imports that Python module to serve the frontend, and this notebook calls the same functions so the notebook and website use one implementation.

> This learning-project assistant is not an emergency service. In a real emergency, follow local authorities and emergency services.

## Before you start

- Install the project's Python dependencies with `python -m pip install -r requirements.txt`.
- Put your Pinecone key in a private `.env` file in the project root. Start with `.env.example`; never paste your real key into this notebook.
- Install and start Ollama, then download the model with `ollama pull qwen2.5:3b`.
- The first index setup downloads the `intfloat/e5-base-v2` embedding model and uploads PDF passages to Pinecone. It needs internet access and can take a few minutes.

Run from the project root or from the `notebooks` folder. The next cell finds the root in either location and configures Django so that it can import the shared RAG functions.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

current_folder = Path.cwd()
if (current_folder / "app" / "config").is_dir():
    PROJECT_DIR = current_folder
elif (current_folder.parent / "app" / "config").is_dir():
    PROJECT_DIR = current_folder.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the project root or its notebooks folder."
    )

load_dotenv(PROJECT_DIR / ".env")
sys.path.insert(0, str(PROJECT_DIR / "app"))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

import django

django.setup()

from observations.rag import (
    PDF_PATH,
    ask_rag,
    create_chunks,
    initialize_rag,
    load_pdf_pages,
    retrieve_documents,
)

print("Project folder:", PROJECT_DIR)
print("Safety PDF:", PDF_PATH)
print("Pinecone key configured:", bool(os.getenv("PINECONE_API_KEY")))

## 1. Read the safety PDF

The PDF reader extracts text page by page. We keep the page number with the text so that retrieved answers can show where the advice came from.

In [ ]:
pages = load_pdf_pages()

print(f"Pages with readable text: {len(pages)}")
for page_number, page_text in pages[:2]:
    print(f"\n--- Page {page_number} ---")
    print(page_text[:1000])

## 2. Split text into smaller passages

A passage, or **chunk**, is a short part of a page. The project uses chunks of up to 500 words with 100 words shared between neighbouring chunks. This overlap helps keep advice together when a topic crosses a chunk boundary.

In [ ]:
chunks = []
for page_number, page_text in pages:
    for chunk in create_chunks(page_text):
        chunks.append({"page": page_number, "text": chunk})

print("Passages created:", len(chunks))
if chunks:
    print("First passage:")
    print(chunks[0]["text"])

## 3. Create or connect to the Pinecone search index

The shared setup function embeds each passage with `intfloat/e5-base-v2` and creates a Pinecone index with 768 dimensions if it does not already exist. It stores vectors in a namespace based on this PDF's content, so a later change to the PDF gets its own set of passages instead of silently reusing stale ones.

This cell needs a valid `PINECONE_API_KEY` in the project root `.env` and internet access. The PDF text and its embeddings are uploaded to Pinecone.

In [ ]:
index, embedding_model, namespace = initialize_rag()
print("Pinecone index is ready.")
print("PDF namespace:", namespace)

## 4. Retrieve relevant passages for a question

The embedding model turns the question into a vector. Pinecone compares that vector with the PDF passages and returns the three closest matches. The E5 model expects the prefix `query:`, which the shared retrieval function adds for us.

In [ ]:
question = "What should I do if I hear rocks falling?"
sources = retrieve_documents(question)

for source in sources:
    print(f"\nPage {source['page']} (similarity {source['score']:.3f})")
    print(source["text"])

## 5. Ask Ollama to answer from the retrieved passages

`ask_rag` performs retrieval and sends the matching passages to the local `qwen2.5:3b` model. The instructions tell the model to stick to the excerpts and say when they do not contain an answer. The result includes the answer and its source passages.

Ollama must be running on this computer and the model must be installed. If the model or Pinecone is unavailable, the function raises a clear error.

In [ ]:
result = ask_rag(question)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(f"\nPage {source['page']} (similarity {source['score']:.3f})")
    print(source["text"])

## How the website uses this RAG system

The React **Safety assistant** page sends a question to Django at `POST /api/safety-assistant/`. Django calls the same `ask_rag` function used above and returns the answer together with its source pages. You do not need to run this notebook every time you ask a question; start the Django and React servers as usual.

The safety guide and the model can be wrong or incomplete. Do not treat a generated response as an official warning or a substitute for advice from local emergency services.